In [1]:
# OOD Dataset Classes Generation
# This script generates the classes.txt file for the OOD dataset
# based on the folder structure in data/ood

import os

# Define the OOD data directory
ood_dir = "data/ood"

# Get all subdirectories (class folders)
class_folders = [d for d in os.listdir(ood_dir) if os.path.isdir(os.path.join(ood_dir, d))]

# Sort the classes alphabetically
class_folders.sort()

# Generate classes.txt with numbered entries
classes_file = os.path.join(ood_dir, "classes.txt")

with open(classes_file, 'w') as f:
    for idx, class_name in enumerate(class_folders):
        # Keep underscores as they are (don't replace with spaces)
        formatted_name = class_name  # Remove the replace() call
        f.write(f"{idx} {formatted_name}\n")

print(f"Generated classes.txt with {len(class_folders)} classes")
print("\nFirst few classes:")
with open(classes_file, 'r') as f:
    for i, line in enumerate(f):
        if i < 10:
            print(line.strip())
        else:
            break

Generated classes.txt with 40 classes

First few classes:
0 black_gram anthracnose
1 black_gram leaf
2 black_gram leaf crinckle
3 black_gram powdery mildew
4 black_gram yellow mosaic
5 chili cercospora
6 chili leaf
7 chili mites and trips
8 chili nutritional deficiency
9 chili powdery mildew


In [2]:
# Verify the generated classes.txt file
print("Complete OOD Classes List:")
print("=" * 50)

with open(classes_file, 'r') as f:
    content = f.read()
    print(content)
    
lines = content.strip().split('\n')
print(f"Total classes: {len(lines)}")

# Show a few examples to verify underscores are preserved
print("\nExamples with underscores preserved:")
for line in lines[:5]:
    print(f"  {line}")

Complete OOD Classes List:
0 black_gram anthracnose
1 black_gram leaf
2 black_gram leaf crinckle
3 black_gram powdery mildew
4 black_gram yellow mosaic
5 chili cercospora
6 chili leaf
7 chili mites and trips
8 chili nutritional deficiency
9 chili powdery mildew
10 guava anthracnose
11 guava canker
12 guava dot
13 guava leaf
14 guava rust
15 guava scab
16 guava styler end root
17 malabar_spinach anthracnose
18 malabar_spinach bacterial spot
19 malabar_spinach downy mildew
20 malabar_spinach leaf
21 malabar_spinach pest damage
22 okra alternaria leaf spot
23 okra cercospora leaf spot
24 okra downy mildew
25 okra leaf
26 okra leaf curly virus
27 onion iris yellow virus
28 onion leaf
29 onion leaf blight
30 onion purple blotch
31 orka phyllosticta leaf spot
32 radish black leaf spot
33 radish downey mildew
34 radish flea beetle
35 radish fresh leaf
36 radish mosaic virus
37 wheat leaf
38 wheat septoria
39 wheat stripe rust

Total classes: 40

Examples with underscores preserved:
  0 black_

In [3]:
# OOD Dataset Sampling Script
# This script samples 3 images per class from the OOD dataset
# and generates metadata with symptoms, causes, and management info

import os
import json
import random
import shutil
from pathlib import Path

# Configuration
ood_dir = "data/ood"
sample_dir = "data/ood_sampled"
metadata_file = "data/ood_sampled/metadata.json"
plant_conditions_file = "data/ood/plant_conditions.json"
samples_per_class = 3
random_seed = 67

# Set random seed for reproducibility
random.seed(random_seed)

# Load plant conditions data
print("Loading plant conditions data...")
with open(plant_conditions_file, 'r') as f:
    plant_conditions = json.load(f)

# Create mapping from class name to condition info
conditions_map = {}
for condition in plant_conditions:
    class_name = condition['class']
    conditions_map[class_name] = condition

print(f"Loaded conditions for {len(conditions_map)} classes")

# Get all class folders
class_folders = [d for d in os.listdir(ood_dir) if os.path.isdir(os.path.join(ood_dir, d))]
class_folders.sort()

print(f"Found {len(class_folders)} classes in OOD directory")

# Create output directory
os.makedirs(sample_dir, exist_ok=True)

# Initialize metadata dictionary
metadata = {}

# Process each class
for class_name in class_folders:
    class_path = os.path.join(ood_dir, class_name)
    
    # Get all image files in the class folder
    image_files = []
    for ext in ['.jpg', '.jpeg', '.png', '.JPG', '.JPEG', '.PNG']:
        image_files.extend([f for f in os.listdir(class_path) if f.lower().endswith(ext.lower())])
    
    if len(image_files) == 0:
        print(f"Warning: No images found in {class_name}")
        continue
    
    # Randomly sample images
    if len(image_files) > samples_per_class:
        sampled_images = random.sample(image_files, samples_per_class)
    else:
        sampled_images = image_files
        print(f"Warning: Only {len(image_files)} images available for {class_name}")
    
    # Create class-specific output folder
    class_sample_dir = os.path.join(sample_dir, class_name)
    os.makedirs(class_sample_dir, exist_ok=True)
    
    # Get condition info for this class
    condition_info = conditions_map.get(class_name, {})
    
    # Copy and rename images
    for idx, image_file in enumerate(sampled_images):
        # Generate new filename with numbering
        new_filename = f"{idx:03d}_{class_name.replace(' ', '_')}.jpg"
        source_path = os.path.join(class_path, image_file)
        dest_path = os.path.join(class_sample_dir, new_filename)
        
        # Copy image
        shutil.copy2(source_path, dest_path)
        
        # Add to metadata
        metadata[new_filename] = {
            "label": class_name.replace(' ', '_').replace('_', ' ').title(),
            "class": class_name,
            "status": condition_info.get('status', 'unknown'),
            "caption": condition_info.get('symptoms', 'No symptoms information available'),
            "cause": condition_info.get('cause', 'No cause information available'),
            "management": condition_info.get('management', condition_info.get('maintenance', 'No management information available')),
            "original_path": os.path.join(class_name, image_file)
        }
    
    print(f"Sampled {len(sampled_images)} images for {class_name}")

# Save metadata
with open(metadata_file, 'w') as f:
    json.dump(metadata, f, indent=2)

print(f"\nSampling complete!")
print(f"Total samples: {len(metadata)}")
print(f"Metadata saved to: {metadata_file}")
print(f"Sampled images saved to: {sample_dir}")

# Display sample metadata
print("\nSample metadata entries:")
for i, (filename, info) in enumerate(list(metadata.items())[:3]):
    print(f"\n{filename}:")
    print(f"  Class: {info['class']}")
    print(f"  Status: {info['status']}")
    print(f"  Caption: {info['caption'][:100]}...")
    print(f"  Cause: {info['cause']}")
    print(f"  Management: {info['management'][:100]}...")
    print(f"  Original: {info['original_path']}")

Loading plant conditions data...
Loaded conditions for 40 classes
Found 40 classes in OOD directory
Sampled 3 images for black_gram anthracnose
Sampled 3 images for black_gram leaf
Sampled 3 images for black_gram leaf crinckle
Sampled 3 images for black_gram powdery mildew
Sampled 3 images for black_gram yellow mosaic
Sampled 3 images for chili cercospora
Sampled 3 images for chili leaf
Sampled 3 images for chili mites and trips
Sampled 3 images for chili nutritional deficiency
Sampled 3 images for chili powdery mildew
Sampled 3 images for guava anthracnose
Sampled 3 images for guava canker
Sampled 3 images for guava dot
Sampled 3 images for guava leaf
Sampled 3 images for guava rust
Sampled 3 images for guava scab
Sampled 3 images for guava styler end root
Sampled 3 images for malabar_spinach anthracnose
Sampled 3 images for malabar_spinach bacterial spot
Sampled 3 images for malabar_spinach downy mildew
Sampled 3 images for malabar_spinach leaf
Sampled 3 images for malabar_spinach pe

In [4]:
# Verify the sampling results
import os
import json

sample_dir = "data/ood_sampled"
metadata_file = "data/ood_sampled/metadata.json"

# Check if sampling was completed
if not os.path.exists(sample_dir):
    print("Sample directory not found. Please run the sampling script first.")
else:
    # Count samples per class
    class_counts = {}
    for class_name in os.listdir(sample_dir):
        class_path = os.path.join(sample_dir, class_name)
        if os.path.isdir(class_path):
            images = [f for f in os.listdir(class_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
            class_counts[class_name] = len(images)
    
    print("Sampling Results:")
    print("=" * 60)
    print(f"Total classes sampled: {len(class_counts)}")
    print(f"Total images: {sum(class_counts.values())}")
    print("\nImages per class:")
    for class_name, count in sorted(class_counts.items()):
        print(f"  {class_name}: {count}")
    
    # Load and display metadata summary
    if os.path.exists(metadata_file):
        with open(metadata_file, 'r') as f:
            metadata = json.load(f)
        
        print(f"\nMetadata entries: {len(metadata)}")
        
        # Show sample entries
        print("\nSample metadata entries:")
        for i, (filename, info) in enumerate(list(metadata.items())[:5]):
            print(f"\n{i+1}. {filename}")
            print(f"   Class: {info['class']}")
            print(f"   Status: {info['status']}")
            print(f"   Caption: {info['caption'][:80]}...")
            print(f"   Original: {info['original_path']}")

Sampling Results:
Total classes sampled: 40
Total images: 120

Images per class:
  black_gram anthracnose: 3
  black_gram leaf: 3
  black_gram leaf crinckle: 3
  black_gram powdery mildew: 3
  black_gram yellow mosaic: 3
  chili cercospora: 3
  chili leaf: 3
  chili mites and trips: 3
  chili nutritional deficiency: 3
  chili powdery mildew: 3
  guava anthracnose: 3
  guava canker: 3
  guava dot: 3
  guava leaf: 3
  guava rust: 3
  guava scab: 3
  guava styler end root: 3
  malabar_spinach anthracnose: 3
  malabar_spinach bacterial spot: 3
  malabar_spinach downy mildew: 3
  malabar_spinach leaf: 3
  malabar_spinach pest damage: 3
  okra alternaria leaf spot: 3
  okra cercospora leaf spot: 3
  okra downy mildew: 3
  okra leaf: 3
  okra leaf curly virus: 3
  onion iris yellow virus: 3
  onion leaf: 3
  onion leaf blight: 3
  onion purple blotch: 3
  orka phyllosticta leaf spot: 3
  radish black leaf spot: 3
  radish downey mildew: 3
  radish flea beetle: 3
  radish fresh leaf: 3
  radis

In [5]:
# Flatten sampled images into single directory
# This moves all images from class folders into one flat directory

import os
import json
import shutil

# Configuration
sample_dir = "data/ood_sampled"
flat_dir = "data/ood_sampled/images"
metadata_file = "data/ood_sampled/metadata.json"
flat_metadata_file = "data/ood_sampled/metadata_flat.json"

# Create flat directory
os.makedirs(flat_dir, exist_ok=True)

# Load original metadata
with open(metadata_file, 'r') as f:
    metadata = json.load(f)

# Initialize flat metadata
flat_metadata = {}

# Process each class folder
class_folders = [d for d in os.listdir(sample_dir) if os.path.isdir(os.path.join(sample_dir, d))]

print(f"Flattening images from {len(class_folders)} classes...")

for class_name in class_folders:
    class_path = os.path.join(sample_dir, class_name)
    
    # Get all images in the class folder
    images = [f for f in os.listdir(class_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    
    for image_file in images:
        source_path = os.path.join(class_path, image_file)
        dest_path = os.path.join(flat_dir, image_file)
        
        # Only copy if destination doesn't exist
        if not os.path.exists(dest_path):
            shutil.copy2(source_path, dest_path)
        else:
            print(f"  Skipping duplicate: {image_file}")
        
        # Update metadata with new path (even if file already existed)
        if image_file in metadata:
            flat_metadata[image_file] = metadata[image_file].copy()
            flat_metadata[image_file]['path'] = os.path.join('images', image_file)
    
    print(f"  Processed {class_name}: {len(images)} images")

# Save flat metadata
with open(flat_metadata_file, 'w') as f:
    json.dump(flat_metadata, f, indent=2)

print(f"\nFlattening complete!")
print(f"Total images in flat directory: {len(flat_metadata)}")
print(f"Flat directory: {flat_dir}")
print(f"Flat metadata saved to: {flat_metadata_file}")

# Verify results
print("\nVerification:")
flat_images = [f for f in os.listdir(flat_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
print(f"Images in {flat_dir}: {len(flat_images)}")

# Show sample of flat metadata
print("\nSample flat metadata entries:")
for i, (filename, info) in enumerate(list(flat_metadata.items())[:3]):
    print(f"\n{filename}:")
    print(f"  Path: {info['path']}")
    print(f"  Class: {info['class']}")
    print(f"  Status: {info['status']}")

Flattening images from 41 classes...
  Skipping duplicate: 000_black_gram_anthracnose.jpg
  Skipping duplicate: 001_black_gram_anthracnose.jpg
  Skipping duplicate: 002_black_gram_anthracnose.jpg
  Processed black_gram anthracnose: 3 images
  Skipping duplicate: 000_black_gram_leaf.jpg
  Skipping duplicate: 001_black_gram_leaf.jpg
  Skipping duplicate: 002_black_gram_leaf.jpg
  Processed black_gram leaf: 3 images
  Skipping duplicate: 000_black_gram_leaf_crinckle.jpg
  Skipping duplicate: 001_black_gram_leaf_crinckle.jpg
  Skipping duplicate: 002_black_gram_leaf_crinckle.jpg
  Processed black_gram leaf crinckle: 3 images
  Skipping duplicate: 000_black_gram_powdery_mildew.jpg
  Skipping duplicate: 001_black_gram_powdery_mildew.jpg
  Skipping duplicate: 002_black_gram_powdery_mildew.jpg
  Processed black_gram powdery mildew: 3 images
  Skipping duplicate: 000_black_gram_yellow_mosaic.jpg
  Skipping duplicate: 001_black_gram_yellow_mosaic.jpg
  Skipping duplicate: 002_black_gram_yellow_m

In [1]:
# Add Cloudflare R2 URLs to metadata
# This script adds image_url field to the flat metadata using R2 public domain

import os
import json

# Configuration
flat_metadata_file = "data/ood_sampled/metadata_flat.json"
updated_metadata_file = "data/ood_sampled/metadata_with_urls.json"

# R2 Configuration
R2_PUBLIC_DOMAIN = os.getenv("R2_PUBLIC_DOMAIN", "https://thesis-assets.andyathsid.com")
R2_PATH_PREFIX = "ood/evaluation/images"

# Load flat metadata
print("Loading flat metadata...")
with open(flat_metadata_file, 'r') as f:
    metadata = json.load(f)

print(f"Loaded {len(metadata)} entries")

# Add image_url field to each entry
updated_metadata = {}
for filename, info in metadata.items():
    # Create the R2 URL
    # The URL format: https://thesis-assets.andyathsid.com/ood/evaluation/images/{filename}
    image_url = f"{R2_PUBLIC_DOMAIN}/{R2_PATH_PREFIX}/{filename}"
    
    # Copy the original info and add the image_url
    updated_info = info.copy()
    updated_info['image_url'] = image_url
    
    updated_metadata[filename] = updated_info

# Save updated metadata
with open(updated_metadata_file, 'w') as f:
    json.dump(updated_metadata, f, indent=2)

print(f"\nUpdated metadata saved to: {updated_metadata_file}")
print(f"Added image_url field to {len(updated_metadata)} entries")

# Display sample entries with URLs
print("\nSample entries with image URLs:")
for i, (filename, info) in enumerate(list(updated_metadata.items())[:3]):
    print(f"\n{i+1}. {filename}")
    print(f"   Class: {info['class']}")
    print(f"   Image URL: {info['image_url']}")
    print(f"   Status: {info['status']}")

# Also update the original flat metadata file with the URLs
with open(flat_metadata_file, 'w') as f:
    json.dump(updated_metadata, f, indent=2)

print(f"\nOriginal flat metadata file also updated with image URLs")
print(f"Total entries: {len(updated_metadata)}")

Loading flat metadata...
Loaded 120 entries

Updated metadata saved to: data/ood_sampled/metadata_with_urls.json
Added image_url field to 120 entries

Sample entries with image URLs:

1. 000_black_gram_anthracnose.jpg
   Class: black_gram anthracnose
   Image URL: https://thesis-assets.andyathsid.com/ood/evaluation/images/000_black_gram_anthracnose.jpg
   Status: diseased

2. 001_black_gram_anthracnose.jpg
   Class: black_gram anthracnose
   Image URL: https://thesis-assets.andyathsid.com/ood/evaluation/images/001_black_gram_anthracnose.jpg
   Status: diseased

3. 002_black_gram_anthracnose.jpg
   Class: black_gram anthracnose
   Image URL: https://thesis-assets.andyathsid.com/ood/evaluation/images/002_black_gram_anthracnose.jpg
   Status: diseased

Original flat metadata file also updated with image URLs
Total entries: 120
